# Random Forest Grid Search


## Setup

The split below matches `train_rf`: `test_size=0.2`, `random_state=42`, and no stratification.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.config import FEATURES, RAW_CSV, SAVED_MODEL_RF, TARGET
from src.model.random_forest import train_rf
from src.utils.file import save_model

## Load Data

In [2]:
df = pd.read_csv(RAW_CSV)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f'Dataset shape: {df.shape}')
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
y_test.value_counts()

Dataset shape: (400, 24)
Train shape: (320, 23)
Test shape: (80, 23)


Type
Typical aura with migraine       49
Migraine without aura            13
Basilar-type aura                 6
Other                             4
Familial hemiplegic migraine      3
Typical aura without migraine     3
Sporadic hemiplegic migraine      2
Name: count, dtype: int64

## Metric Helper

In [3]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }
    return metrics, y_pred

## Baseline Model

This calls `train_rf(..., save=False)` so the baseline behavior comes from the project model code without overwriting any saved model before tuning.

In [4]:
baseline_model, baseline_accuracy = train_rf(df, save=False)
baseline_metrics, baseline_pred = evaluate_model(baseline_model, X_test, y_test)

pd.DataFrame([baseline_metrics], index=['baseline']).T

Accuracy: 0.925


,baseline
accuracy,0.925000
precision_macro,0.808571
recall_macro,0.747085
f1_macro,0.769821
precision_weighted,0.916333
recall_weighted,0.925000
f1_weighted,0.917689


In [5]:
print(classification_report(y_test, baseline_pred, zero_division=0))
labels = sorted(y_test.unique())
pd.DataFrame(confusion_matrix(y_test, baseline_pred, labels=labels), index=labels, columns=labels)

                               precision    recall  f1-score   support

            Basilar-type aura       0.83      0.83      0.83         6
 Familial hemiplegic migraine       1.00      0.67      0.80         3
        Migraine without aura       0.87      1.00      0.93        13
                        Other       1.00      0.75      0.86         4
 Sporadic hemiplegic migraine       0.00      0.00      0.00         2
   Typical aura with migraine       0.96      0.98      0.97        49
Typical aura without migraine       1.00      1.00      1.00         3

                     accuracy                           0.93        80
                    macro avg       0.81      0.75      0.77        80
                 weighted avg       0.92      0.93      0.92        80



,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,5,0,0,0,0,1,0
Familial hemiplegic migraine,0,2,1,0,0,0,0
Migraine without aura,0,0,13,0,0,0,0
Other,0,0,1,3,0,0,0
Sporadic hemiplegic migraine,1,0,0,0,0,1,0
Typical aura with migraine,0,0,0,0,1,48,0
Typical aura without migraine,0,0,0,0,0,0,3


## Grid Search

The grid optimizes weighted F1 because the target classes are imbalanced. The final comparison is still done on the untouched holdout test set.

In [6]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print(f'Best CV weighted F1: {grid_search.best_score_:.4f}')
grid_search.best_params_

Best CV weighted F1: 0.8706


{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100}

## Tuned Model Comparison

In [7]:
tuned_model = grid_search.best_estimator_
tuned_metrics, tuned_pred = evaluate_model(tuned_model, X_test, y_test)

comparison = pd.DataFrame([baseline_metrics, tuned_metrics], index=['baseline', 'tuned']).T
comparison['delta'] = comparison['tuned'] - comparison['baseline']
comparison

,baseline,tuned,delta
accuracy,0.925000,0.925000,0.0
precision_macro,0.808571,0.808571,0.0
recall_macro,0.747085,0.747085,0.0
f1_macro,0.769821,0.769821,0.0
precision_weighted,0.916333,0.916333,0.0
recall_weighted,0.925000,0.925000,0.0
f1_weighted,0.917689,0.917689,0.0


In [8]:
print(classification_report(y_test, tuned_pred, zero_division=0))
pd.DataFrame(confusion_matrix(y_test, tuned_pred, labels=labels), index=labels, columns=labels)

                               precision    recall  f1-score   support

            Basilar-type aura       0.83      0.83      0.83         6
 Familial hemiplegic migraine       1.00      0.67      0.80         3
        Migraine without aura       0.87      1.00      0.93        13
                        Other       1.00      0.75      0.86         4
 Sporadic hemiplegic migraine       0.00      0.00      0.00         2
   Typical aura with migraine       0.96      0.98      0.97        49
Typical aura without migraine       1.00      1.00      1.00         3

                     accuracy                           0.93        80
                    macro avg       0.81      0.75      0.77        80
                 weighted avg       0.92      0.93      0.92        80



,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,5,0,0,0,0,1,0
Familial hemiplegic migraine,0,2,1,0,0,0,0
Migraine without aura,0,0,13,0,0,0,0
Other,0,0,1,3,0,0,0
Sporadic hemiplegic migraine,1,0,0,0,0,1,0
Typical aura with migraine,0,0,0,0,1,48,0
Typical aura without migraine,0,0,0,0,0,0,3


## Save Tuned Model

Set `SAVE_TUNED_MODEL = False` if you only want to inspect the results without replacing the saved random forest model.

In [9]:
SAVE_TUNED_MODEL = True

if SAVE_TUNED_MODEL:
    save_model(tuned_model, SAVED_MODEL_RF)
    print(f'Saved tuned model as {SAVED_MODEL_RF}')

Saved tuned model as random_forest.pkl
